In [ ]:
# Load .env
from dotenv import load_dotenv
import os

load_dotenv()
DATA_DIR = os.getenv("DATA_DIR")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

In [ ]:
# Download the Telstra dataset

import zipfile
from kaggle.api.kaggle_api_extended import KaggleApi

# Initialize and authenticate the API
api = KaggleApi()
api.authenticate()

# Define the competition name
competition_name = "telstra-recruiting-network"

# Download all files associated with the competition
# This downloads a single .zip file to the specified path
api.competition_download_files(competition_name, path=DATA_DIR)

# Unzip the downloaded dataset
zip_path = f"{DATA_DIR}/{competition_name}.zip"
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

# Unzip all files in DATA_DIR
for item in os.listdir(DATA_DIR):
    if item.endswith(".zip"):
        file_path = os.path.join(DATA_DIR, item)
        with zipfile.ZipFile(file_path, "r") as zip_ref:
            zip_ref.extractall(DATA_DIR)

print("Download and extraction complete!")

Download and extraction complete!


In [ ]:
# Load all Telstra recruiting network data files
import pandas as pd
import os

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
severity = pd.read_csv(os.path.join(DATA_DIR, "severity_type.csv"))
resource = pd.read_csv(os.path.join(DATA_DIR, "resource_type.csv"))
log = pd.read_csv(os.path.join(DATA_DIR, "log_feature.csv"))
event = pd.read_csv(os.path.join(DATA_DIR, "event_type.csv"))

In [4]:
# Merge all auxiliary tables on 'id' to create unified datasets
# Many-to-one relationships exist for event_type, log_feature, and resource_type
train_df = (
    train.merge(severity, on="id", how="left")
    .merge(resource, on="id", how="left")
    .merge(log, on="id", how="left")
    .merge(event, on="id", how="left")
)

test_df = (
    test.merge(severity, on="id", how="left")
    .merge(resource, on="id", how="left")
    .merge(log, on="id", how="left")
    .merge(event, on="id", how="left")
)

print("Unified Train dataframe shape:", train_df.shape)
print("Unified Test dataframe shape:", test_df.shape)

Unified Train dataframe shape: (61839, 8)
Unified Test dataframe shape: (84584, 7)


## The graph schema

Nodes:

`:DisruptionEvent`: The core entity (representing the id column). This is your target node.
`:Location`: The physical or logical site where the event occurred.
`:LogFeature`: The specific error codes or log strings generated.  
`:EventType`: The high-level classification of what happened.
`:ResourceType`: The specific part of the network infrastructure affected (e.g., router, switch, fiber optic link).
`:SeverityType`: The internal warning level of the log message itself.  

Relationships:

`(:DisruptionEvent) -[:OCCURRED_AT]-> (:Location)`
`(:DisruptionEvent) -[:LOGGED {volume: Int}]-> (:LogFeature)` (Note: The edge holds the volume property from the data).
`(:DisruptionEvent) -[:TRIGGERED]-> (:EventType)`
`(:DisruptionEvent) -[:ON_RESOURCE]-> (:ResourceType)`
`(:DisruptionEvent) -[:HAS_ALARM]-> (:SeverityType)`
`(:Location) -[:NEXT_EVENT]-> (:Location)` (If id: 500 occurs at Location A and id: 501 occurs at Location B, draw a directed edge from A→B. If this sequence happens repeatedly across the dataset, increase the edge weight. This acts as a proxy for how faults cascade through the network.)